# 04 — Train / Test split

**Time-based split** (no random shuffle — football is temporal):
- **Train**: matches before 2018-01-01 (~80%)
- **Test**:  matches from 2018-01-01 to today (~20%)

We also drop the earliest matches where Elo hasn't converged.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
df = pd.read_parquet(ROOT / 'data' / 'processed' / 'matches_features.parquet')
df = df[df.date >= '1950-01-01'].reset_index(drop=True)  # let Elo settle
print('rows after burn-in:', len(df))


In [ ]:
FEATURES = [
    'home_elo','away_elo','elo_diff',
    'home_form5_pts','home_form5_gf','home_form5_ga',
    'away_form5_pts','away_form5_gf','away_form5_ga',
    'home_form10_pts','home_form10_gf','home_form10_ga',
    'away_form10_pts','away_form10_gf','away_form10_ga',
    'h2h_home_wins','h2h_draws','h2h_away_wins','h2h_home_gf','h2h_home_ga',
    'tournament_k','neutral_int','home_rest','away_rest',
]
TARGET_CLS = 'target'        # 0=home,1=draw,2=away
TARGET_HGOALS = 'home_score'
TARGET_AGOALS = 'away_score'

df = df.dropna(subset=FEATURES).reset_index(drop=True)
print('rows after dropna:', len(df))


In [ ]:
CUTOFF = '2018-01-01'
train = df[df.date <  CUTOFF].copy()
test  = df[df.date >= CUTOFF].copy()
print(f'train: {len(train):,}  test: {len(test):,}')
print('train target dist:\n', train.target.value_counts(normalize=True).round(3))
print('test  target dist:\n', test.target.value_counts(normalize=True).round(3))


In [ ]:
OUT = ROOT / 'data' / 'train_test'
OUT.mkdir(parents=True, exist_ok=True)
cols = ['date','home_team','away_team','tournament','neutral'] + FEATURES + [TARGET_CLS, TARGET_HGOALS, TARGET_AGOALS]
train[cols].to_parquet(OUT / 'train.parquet', index=False)
test[cols].to_parquet(OUT / 'test.parquet', index=False)
pd.Series(FEATURES).to_csv(OUT / 'feature_list.csv', index=False, header=['feature'])
print('wrote', OUT)
